In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import cmudict, wordnet

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('cmudict', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)


True

In [2]:
def preprocess_text(text):
    
    sentences = sent_tokenize(text)
    processed_corpus = []
    
    for sent in sentences:
        tokens = word_tokenize(sent)
        pos_tags = pos_tag(tokens)
        processed_corpus.append({
            "sentence": sent,
            "tokens": tokens,
            "pos_tags": pos_tags
        })
        
    return processed_corpus


sample_text = "Peter Piper picked a peck of pickled peppers. Her smile was bright like the sun."
print(preprocess_text(sample_text))

[{'sentence': 'Peter Piper picked a peck of pickled peppers.', 'tokens': ['Peter', 'Piper', 'picked', 'a', 'peck', 'of', 'pickled', 'peppers', '.'], 'pos_tags': [('Peter', 'NNP'), ('Piper', 'NNP'), ('picked', 'VBD'), ('a', 'DT'), ('peck', 'NN'), ('of', 'IN'), ('pickled', 'JJ'), ('peppers', 'NNS'), ('.', '.')]}, {'sentence': 'Her smile was bright like the sun.', 'tokens': ['Her', 'smile', 'was', 'bright', 'like', 'the', 'sun', '.'], 'pos_tags': [('Her', 'PRP$'), ('smile', 'NN'), ('was', 'VBD'), ('bright', 'VBN'), ('like', 'IN'), ('the', 'DT'), ('sun', 'NN'), ('.', '.')]}]


In [3]:
def detect_similes(pos_tags):
    
    similes = []
    tokens = [t[0] for t in pos_tags]
    tags = [t[1] for t in pos_tags]
    
    for i in range(len(pos_tags)):
        word = tokens[i].lower()
        
       
        if word == 'like' and i + 1 < len(pos_tags):
            similes.append({
                "figure": "Simile",
                "matched_pattern": f"like {tokens[i+1]}",
                "index": i
            })
      
        elif word == 'as' and i + 2 < len(pos_tags):
            if tokens[i+2].lower() == 'as' and tags[i+1].startswith('JJ'):
                similes.append({
                    "figure": "Simile",
                    "matched_pattern": f"as {tokens[i+1]} as",
                    "index": i
                })
                
    return similes

In [5]:
cmu_dict = cmudict.dict()

def get_initial_phoneme(word):
    """Extracts the first phoneme sound of a word using CMU dict."""
    word = word.lower()
    if word in cmu_dict:
        
        return cmu_dict[word][0][0]
    return None

def detect_alliteration(tokens, min_consecutive=2):
    
    alliterations = []
    current_streak = []
    current_phoneme = None
    
    for word in tokens:
        if not word.isalpha():
            continue
        
        phoneme = get_initial_phoneme(word)
        if not phoneme:
            continue
            
        if phoneme == current_phoneme:
            current_streak.append(word)
        else:
            if len(current_streak) >= min_consecutive:
                alliterations.append({
                    "figure": "Alliteration",
                    "matched_phrase": " ".join(current_streak)
                })
            current_streak = [word]
            current_phoneme = phoneme
     
    if len(current_streak) >= min_consecutive:
        alliterations.append({
            "figure": "Alliteration",
            "matched_phrase": " ".join(current_streak)
        })
        
    return alliterations

In [6]:
KNOWN_OXYMORONS = {
    ("deafening", "silence"), ("jumbo", "shrimp"), ("open", "secret"),
    ("bittersweet", ""), ("living", "dead"), ("original", "copy"),
    ("act", "naturally"), ("awfully", "good"), ("virtual", "reality")
}

def check_wordnet_antonym(word1, word2):
   
    syns1 = wordnet.synsets(word1)
    syns2 = wordnet.synsets(word2)
    
    for s1 in syns1:
        for lemma in s1.lemmas():
            for ant in lemma.antonyms():
                ant_name = ant.name().split('.')[0]
                for s2 in syns2:
                    if ant_name == s2.name().split('.')[0]:
                        return True
    return False

def detect_oxymorons(pos_tags):
 
    oxymorons = []
    tokens = [t[0] for t in pos_tags]
    tags = [t[1] for t in pos_tags]
    
    for i in range(len(pos_tags) - 1):
        w1, t1 = tokens[i].lower(), tags[i]
        w2, t2 = tokens[i+1].lower(), tags[i+1]
     
        if (w1, w2) in KNOWN_OXYMORONS:
            oxymorons.append({
                "figure": "Oxymoron",
                "matched_phrase": f"{tokens[i]} {tokens[i+1]}",
                "method": "Lexicon Match"
            })
            continue
            
      
        if (t1.startswith('JJ') and t2.startswith('NN')) or (t1.startswith('JJ') and t2.startswith('JJ')):
            if check_wordnet_antonym(w1, w2):
                oxymorons.append({
                    "figure": "Oxymoron",
                    "matched_phrase": f"{tokens[i]} {tokens[i+1]}",
                    "method": "WordNet Antonym"
                })
                
    return oxymorons

In [7]:
ONOMATOPOEIA_LEXICON = {
    "bang", "boom", "buzz", "click", "clash", "crash", "hiss", "hum", 
    "mew", "moo", "pop", "quack", "roar", "rustle", "screech", "slap", 
    "splash", "squeak", "thud", "thump", "tick", "whack", "whisper", 
    "zap", "zoom", "zip", "giggle", "murmur", "snort", "splat", "bam"
}

def detect_onomatopoeia(tokens):
    """
    Detects onomatopoeic words by filtering tokens against a phonetic acoustic lexicon.
    """
    matches = []
    for word in tokens:
        clean_word = word.lower().strip()
        if clean_word in ONOMATOPOEIA_LEXICON:
            matches.append({
                "figure": "Onomatopoeia",
                "matched_phrase": word,
                "method": "Lexicon Match"
            })
    return matches

In [8]:
def analyze_paragraph_full(paragraph):
    corpus = preprocess_text(paragraph)
    full_results = []
    
    for item in corpus:
        similes = detect_similes(item["pos_tags"])
        alliterations = detect_alliteration(item["tokens"])
        oxymorons = detect_oxymorons(item["pos_tags"])
        onomatopoeia = detect_onomatopoeia(item["tokens"])
        
        combined_detections = similes + alliterations + oxymorons + onomatopoeia
        
        full_results.append({
            "sentence": item["sentence"],
            "detections": combined_detections
        })
        
    return full_results

test_paragraph = (
    "The loud silence filled the room while a sharp buzz echoed. "
    "Peter Piper picked peppers, and the soldier fought like a lion."
)

results = analyze_paragraph_full(test_paragraph)

for res in results:
    print(f"Sentence: {res['sentence']}")
    if res['detections']:
        for d in res['detections']:
            match_str = d.get('matched_pattern') or d.get('matched_phrase')
            print(f"  -> Detected [{d['figure']}]: '{match_str}' (Engine: {d.get('method', 'POS Rule')})")
    else:
        print("  -> No figures of speech matched.")
    print("-" * 60)

Sentence: The loud silence filled the room while a sharp buzz echoed.
  -> Detected [Onomatopoeia]: 'buzz' (Engine: Lexicon Match)
------------------------------------------------------------
Sentence: Peter Piper picked peppers, and the soldier fought like a lion.
  -> Detected [Simile]: 'like a' (Engine: POS Rule)
  -> Detected [Alliteration]: 'Peter Piper picked peppers' (Engine: POS Rule)
------------------------------------------------------------
